In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType,
    FloatType,
)
import pyspark.sql.functions as F
catalog_name = 'ecommerce'

In [0]:
df_bronze_brands=spark.read.table(f'{catalog_name}.bronze.brz_brands')
df_bronze_brands.show(10)


In [0]:
# remove the extraspace from the brand_name
df_silver=df_bronze_brands.withColumn('brand_name',F.trim(F.col("brand_name")))
df_silver.show()

In [0]:
# remove extra characters from the brand_name regexp(col,checkcahetr, replace)
df_silver=df_silver.withColumn('brand_code',F.regexp_replace(F.col("brand_code"),r'[^A-Za-z0-9]',''))
df_silver.show()


In [0]:
df_silver.select('category_code').distinct().show()


In [0]:
# we can see different category with same intention like toy toys
anamolyis={
    'GROCERY':"GRCY",
    'BOOKS':'BKS',
    'TOYS':'TOY'
}
df_silver=df_silver.replace(anamolyis,subset=["category_code"])
df_silver.select('category_code').distinct().show()

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_brands')

### **Category**

In [0]:
df_bronze_category=spark.read.table(f'{catalog_name}.bronze.brz_category')
df_bronze_category.show()

In [0]:
df_silver_category=df_bronze_category.select("category_code").groupBy('category_code').count()
df_silver_category.show()

In [0]:
df_silver_category=df_bronze_category.dropDuplicates(['category_code'])
df_silver_category=df_bronze_category.withColumn('category_code',F.upper(F.col("category_code")))

df_silver_category.show(truncate=False)

In [0]:
df_silver_category.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_category')